In [0]:
from openai import OpenAI
import re
ctx=dbutils.notebook.entry_point.getDbutils().notebook().getContext()
client=OpenAI(
    api_key=ctx.apiToken().get(),
    base_url=f"https://{ctx.browserHostName().get()}/serving-endpoints",
)

model="databricks-meta-llama-3-3-70b-instruct"

System="""You are a Spark SQL expert. Convert the user's question into one SQL query.

Tables:
chat_bot.gold.sales_enriched (one row per sale)
sale_id, sale_date (date), sale_month (string yyyy-MM), customer_id, customer_name,
region (North/South/East/West), segment (Retail/Corporate/Online),
product_id, product_name, category (Laptops/Phones/Accessories/Audio),
price (unit price), quantity, total_amount (revenue of the sale)

chat_bot.gold.monthly_sales_by_category
  sale_month (string yyyy-MM), category, revenue, units_sold, orders


Data covers 2026-04 to 2026-09.

Rules:
- Use only the tables and columns above, with full names like chat_bot.gold.sales_enriched
- Only SELECT queries
- Add LIMIT 50 unless the query already limits rows or returns one aggregate row
- Return only the SQL, with no explanation and no markdown
"""

def clean_Sql(text):
    text=text.strip()
    if text.startswith("```"):
        text=text.strip("`")
        if text.lower().startswith("sql"):
            text=text[3:]
    return text.strip().rstrip(";")            



def generate_Sql(text):
    resp=client.chat.completions.create(
    model=model,
    messages=[{"role":"system","content":System},{"role":"user","content":text}],
    temperature=0.1,
    max_tokens=1000 
    )
    return clean_Sql(resp.choices[0].message.content)


BLOCKED = ["insert", "update", "delete", "drop", "alter", "create", "truncate", "merge", "grant"]

def is_Safe(sql):
    s=sql.strip().lower()
    if not (s.startswith("select") or s.startswith("with")):
        return False
    if ";" in s:
        return False
    return not any(re.search(rf"\b{w}\b", s) for w in BLOCKED)

def tables_ok(sql):
    refs = re.findall(r"(?:from|join)\s+([\w.]+)", sql, flags=re.I)
    return all(r.lower().startswith("chat_bot.gold.") for r in refs)

def run_Sql(sql):
    if not(is_Safe(sql)and  tables_ok(sql)):
        raise ValueError(f"SQL is not safe:{sql}")   
    return spark.sql(sql).toPandas()

for q in ["What is the total revenue?", "Top 5 products by revenue","What is the total revenue by region?"]:

    sql = generate_Sql(q)
    print(sql)
    print(run_Sql(sql))
    print("-" * 50)

try:
    run_Sql("DROP TABLE chat_bot.gold.sales_enriched")
except ValueError as e:
    print("Blocked:", e)





In [0]:
# bad_sql = "SELECT region, SUM(revenue) FROM chat_bot.gold.sales_enriched GROUP BY region"
# run_Sql(bad_sql)


def fix_Sql(question,bad_sql,error):
    resp=client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":System},
            {"role":"user","content":f"""Questions:{question}
             
             This SQL Failed
             {bad_sql}

             Error: {error}

             Return a Corrected Query Only"""
             
}],
        temperature=0,
        max_tokens=3000
        
        )
    
    return clean_Sql(resp.choices[0].messages.content)

def ask_question(question,max_tried=2):
    sql=generate_Sql(question)
    for attempt in range(max_tried+1):    
        try:
            return run_Sql(sql)
        except Exception as e:
            if attempt==max_tried:
                return f"Sql:Failed:{e}"
            sql=fix_Sql(question,sql,e)    

ask_question("What is the total revenue by country")            
        










    


In [0]:
import json
def summarize(question,sql,df):
    data=df.head(20).to_csv(index=False)
    resp=client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":"You answer business questions using only the data provided."
                                           "Be brief and clear. Do not invent numbers. "
                                           "If the data is empty, say no data was found."},
            {"role": "user", "content": f"Question: {question}\n\nData (CSV):\n{data}"}

        ],
        temperature=0,
        max_tokens=3000
    )
    return resp.choices[0].message.content


def chat_question(q):
     sql=generate_Sql(q)
     result=ask_question(q)
     if isinstance(result, str):
       return {"answer": "Sorry, I could not answer that from the sales data.", "sql": sql, "rows": []}
     rows = json.loads(result.head(20).to_json(orient="records", date_format="iso"))
     return {"answer": summarize(q, sql, result), "sql": sql, "rows": rows}
     

for q in ["What is the total revenue?", "Top 5 products by revenue","What is the total revenue by region?",     "Monthly revenue by region limit to 5","is there any customer34"]:
    print(chat_question(q))

   




In [0]:
import json
import pandas as pd

def chart_Selection(df,question):
    CHART_SYSTEM = """You choose a chart for a SQL query result.
    Question:{question}
    Return only JSON with keys: chart_type, x, y, series, title.
    - chart_type: "bar", "line", "pie" or "none"
    -line chart only for trends
    - x: the text or date column for the labels. y: the numeric column.
    -Only use series when the user explicitly asks for a breakdown by a second dimension.
     For "Total Revenue by Product", series must be null.
    - x, y and series must be copied exactly from the column names provided in the user message. Never use generic words like value, category or label
    - title: Create the title directly from the user's {question}
    
     """

    resp=client.chat.completions.create(
        model=model,
        messages=[
            {"role":"user","content":sql,
             "role":"system","content":CHART_SYSTEM
             }
        ],
        temperature=0,
        max_tokens=1000
    )
    result=clean_Sql(resp.choices[0].message.content)
    result=result.replace("json","",1)
    spec=json.loads(result)


    nums_col=df.select_dtypes("number").columns.to_list()
    text_col=[c for c in df.columns if c not in nums_col]

    if spec.get("y") not in df.columns and nums_col:
        spec['y']=nums_col[0]
    if spec.get("x") not in df.columns and text_col:
        spec['x']=text_col[0]
    ok=(
        spec['x'] in df.columns and spec['y'] in df.columns)
    
    if not ok:
        return None
    print(spec)
    return visual_plot(df,spec)

def visual_plot(df,spec):
    import matplotlib.pyplot as plt
    if spec['chart_type']==None:
        return None
    x,y,s,kind=spec['x'],spec['y'],spec['series'],spec['chart_type']
    d=df.copy()
    fig,ax= plt.subplots(figsize=(8,4))

    if kind=="pie":
       ax.pie(d[y],labels=d[x].astype(str))

    else:
        if s:
            d.pivot(index=x, columns=s, values=y).plot(kind=kind, ax=ax, marker="o" if kind == "line" else None)
        elif kind == "bar":
            ax.bar(d[x].astype(str), d[y])
            print("yes")
        else:
            ax.plot(d[x].astype(str), d[y], marker="o") 
    plt.xticks(rotation=45, ha="right")
    ax.set_title(spec.get("title", ""))
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    plt.tight_layout()
    plt.show()            

    

question="No of product sold by category"
result=ask_question(question)
print(result)
print(chart_Selection(result,question))

    













In [0]:

tests = [
    ("What is the total revenue?",
     "SELECT SUM(total_amount) FROM chat_bot.gold.sales_enriched"),
    ("Top 5 products by revenue",
     "SELECT product_name, SUM(total_amount) AS revenue FROM chat_bot.gold.sales_enriched GROUP BY product_name ORDER BY revenue DESC LIMIT 5"),
    ("Total revenue by region",
     "SELECT region, SUM(total_amount) FROM chat_bot.gold.sales_enriched GROUP BY region"),
    ("Monthly revenue",
     "SELECT sale_month, SUM(total_amount) FROM chat_bot.gold.sales_enriched GROUP BY sale_month ORDER BY sale_month"),
    ("How many orders were placed in 2026-08?",
     "SELECT COUNT(*) FROM chat_bot.gold.sales_enriched WHERE sale_month = '2026-08'"),
    ("Which category sold the most units?",
     "SELECT category, SUM(quantity) AS units FROM chat_bot.gold.sales_enriched GROUP BY category ORDER BY units DESC LIMIT 1")
]

def check_sql(q):
    sql=generate_Sql(q)
    return sql

def check_Query_Comparasion(test,expected):
    System="""You are a SQL query evaluator.

Your task is to determine whether a MODEL-GENERATED SQL query is semantically equivalent to an EXPECTED/TEST SQL query.

Do NOT compare the queries as exact strings.

Ignore differences that do not change the meaning or result, including:
- Column aliases such as AS total_revenue
- Whitespace and formatting
- Capitalization
- SQL keyword casing
- Unnecessary LIMIT clauses when they do not change the intended result
- Equivalent SQL formatting or syntax

Focus on whether both queries answer the same user question and produce the same intended result.

Evaluate:
1. Tables used
2. Columns used
3. Filters
4. JOIN conditions
5. Aggregations
6. GROUP BY
7. ORDER BY when relevant to the requested answer
8. LIMIT when it materially changes the answer
9. Overall semantic meaning
Expected:{expected}
Test:{Test}
Return Anwer only in Yes or No, nothing Else
    """
    resp= client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":System},
            {"role":"user","content":test}
        ],
        temperature=0,
        max_tokens=1000

    )

    return resp.choices[0] .message.content

In [0]:
total_passed=0
for i in tests:
    expected_result=generate_Sql(i[0])
    test_result=i[1]
    result=check_Query_Comparasion(test_result,expected_result)
    if result=="Yes":
        total_passed+=1
print(f"Accuracy:{total_passed}/{len(tests)}")        



    

In [0]:
spark.sql("select * from chat_bot.gold.sales_enriched limit 5").display()
